# Do Vulnerability Detectors Learn the Bug or the Names?
**TF-IDF with Random Forest versus fine-tuned CodeBERT on C functions**

ARTI 404 Natural Language Processing · IAU (CCSIT) · Section 9MA2 · Group 1
Khalid Waleed Alhilal (leader) · Ali Khalid Albinali · Mishary Waleed AlAwadh

**Question.** Which method detects vulnerable C functions more reliably, and which one loses less accuracy
and macro F1 when identifier names are replaced?

**Input / output.** One C function as text → `Vulnerable` or `Safe` and the probability of `Vulnerable`.

## 1. Setup
Environment: Jupyter (local), Python 3.11, CUDA 12.8, NVIDIA RTX 5080.

`RUN_FULL_TRAINING = False` (default) loads the saved models and results and checks that the seed-42 test
metrics recomputed from the loaded models equal `results/metrics_summary.json`.
`True` retrains everything (several hours on a GPU).

In [ ]:
RUN_FULL_TRAINING = False

import json, sys, platform
from pathlib import Path
REPO = Path.cwd()
sys.path.insert(0, str(REPO))

In [ ]:
# Package versions
import importlib.metadata as md
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
for pkg in ["transformers", "datasets", "scikit-learn", "pandas", "numpy", "tree-sitter", "tree-sitter-c", "scipy", "matplotlib"]:
    print(f"{pkg:<14}", md.version(pkg))

## 2. Data
CodeXGLUE Defect Detection (Devign): C functions from FFmpeg and QEMU, label 1 = vulnerable.
Cleaning removes comments and extra whitespace; each function is hashed so that copies of the same
function stay in one split only.

In [ ]:
# TODO (C03): load data/processed splits, show data_stats.json, class counts, duplicates, truncation at 512 tokens

### Renamed test set
User-defined variable and function names are replaced with `VAR1, VAR2, …` and `FUN1, FUN2, …` (tree-sitter).
Keywords, library calls, struct fields, type names and ALL_CAPS macros are kept, so the label stays valid.

In [ ]:
# TODO (C03): renaming stats + before/after example

## 3. Method 1 — TF-IDF + Random Forest
Own C tokenizer (operators kept whole, identifiers split into sub-parts) → TF-IDF unigrams + bigrams →
Random Forest. Hyper-parameters chosen on dev macro F1 only.

In [ ]:
# TODO (C03): tokenizer example, dev grid table, best params, load or train models

## 4. Method 2 — fine-tuned CodeBERT
`microsoft/codebert-base` + classification head, max 512 tokens, AdamW with linear schedule, bf16,
learning rate and epoch chosen on dev macro F1 with early stopping.

In [ ]:
# TODO (C03): LR search table, training curves, load or train models

## 5. Results
All numbers below are read from `results/metrics_summary.json`.

In [ ]:
# TODO (C03): results table (original vs renamed, mean ± std over 3 seeds), majority baseline,
# McNemar, confusion matrices, drop chart, truncation table, augmentation, reproducibility check

## 6. Try it
Paste one C function into `code` and run the cell. The same cleaning as the training data is applied;
no retraining happens.

In [ ]:
from src.predict import try_it

code = """
void copy_name(char *src) {
    char buf[64];
    strcpy(buf, src);
}
"""
try_it(code)

### Original examples (written by the team, not part of the test score)

In [ ]:
# TODO (C03): predictions on data/original_examples.json

## 7. Sources
- Dataset: S. Lu et al., "CodeXGLUE," NeurIPS Datasets and Benchmarks, 2021 · Y. Zhou et al., "Devign," NeurIPS, 2019. Hugging Face: `google/code_x_glue_cc_defect_detection`.
- Pretrained model: `microsoft/codebert-base` · Z. Feng et al., "CodeBERT," Findings of EMNLP, 2020.
- Parsing: tree-sitter, tree-sitter-c.
- Libraries: PyTorch, Hugging Face Transformers and Datasets, scikit-learn, pandas, NumPy, SciPy, statsmodels, Matplotlib.
- Papers: [1] Lu et al. 2021 · [2] Zhou et al. 2019 · [3] Russell et al., ICMLA 2018 · [4] Feng et al. 2020 · [5] Li et al., VulDeePecker, NDSS 2018 · [6] Risse and Böhme, USENIX Security 2024 · [7] Fu and Tantithamthavorn, LineVul, MSR 2022 · [8] Chakraborty et al., IEEE TSE 2022.